In [0]:
# Importa o módulo "os", que permite interagir com o sistema operacional
# (caminhos de pastas, listar arquivos, variáveis de ambiente etc.)
import os

# os.getcwd() = "get current working directory" (obtém o diretório de trabalho atual).
# Retorna, como texto, o caminho da pasta onde o notebook está sendo executado.
# print() mostra esse caminho na saída da célula.
print(os.getcwd())

In [0]:
# IMPORT DOS DADOS

# spark.read.csv(...) lê um arquivo CSV e devolve um DataFrame do Spark
# (estrutura distribuída, processada em cluster).
# O argumento é o caminho do arquivo/pasta a ser lido.
# .toPandas() converte o DataFrame do Spark em um DataFrame do pandas,
# que fica inteiro na memória do driver (bom para bases pequenas/médias,
# arriscado para bases muito grandes).
df = spark.read.csv("/Workspace/Users/patrianegovi@gmail.com/machine_learning_curso_asn").toPandas()

# df.info() mostra um resumo do DataFrame: número de linhas, nome e tipo
# de cada coluna, quantidade de valores não nulos.
# memory_usage='deep' faz o pandas medir o consumo real de memória,
# inclusive de colunas de texto (object), que por padrão são subestimadas.
df.info(memory_usage='deep')

In [0]:
# IMPORT DOS DADOS

# spark.read.csv(...) lê o CSV e devolve um DataFrame do Spark (distribuído, processado no cluster).
# .toPandas() converte para um DataFrame do pandas, que fica inteiro na memória do driver.
# É prático para bases pequenas/médias; em bases muito grandes pode estourar a memória.
df = spark.read.csv("/Workspace/Users/patrianegovi@gmail.com/machine_learning_curso_asn").toPandas()

# df.head() mostra as 5 primeiras linhas do DataFrame.
# É a forma rápida de conferir se a leitura deu certo: nomes das colunas,
# formato dos valores, se o separador foi reconhecido etc.
# df.head(10) mostra as 10 primeiras; df.tail() mostra as últimas.
df.head()

In [0]:
# dbutils.fs.ls(caminho) lista o conteúdo de uma pasta no sistema de arquivos do Databricks.
# Aqui, o caminho é o Volume do Unity Catalog, que segue o padrão:
# /Volumes/<catalogo>/<schema>/<volume>
#   catalogo = olist | schema = default | volume = abt_ativacao
# Retorna uma lista com um item por arquivo/pasta, contendo:
#   path (caminho completo), name (nome), size (tamanho em bytes), modificationTime (última modificação).

# display(...) mostra o resultado como uma tabela interativa no notebook
# (ordenável, com opção de baixar). Sem o display, o print mostraria uma lista crua e difícil de ler.
display(dbutils.fs.ls("/Volumes/olist/default/abt_ativacao"))

In [0]:
# Os parênteses externos permitem encadear várias chamadas em linhas separadas
# (encadeamento de métodos), sem precisar de "\" no fim de cada linha.
(spark.read.csv(
    # Caminho absoluto do CSV dentro do Volume do Unity Catalog.
    "/Volumes/olist/default/abt_ativacao/abt_ativacao_pati.csv",

    # header=True: a primeira linha do arquivo vira o nome das colunas.
    # inferSchema=True: o Spark lê os dados e detecta o tipo de cada coluna
    # (inteiro, decimal, texto...). Sem isso, tudo viria como string.
    header=True, inferSchema=True)

 # .write: inicia a escrita do DataFrame (agora o fluxo é de saída, não de leitura).
 # .mode("overwrite"): se a tabela já existir, substitui o conteúdo.
 # Outros modos: "append" (acrescenta linhas), "ignore" (não faz nada se existir),
 # "error" (padrão, dá erro se existir).
 .write.mode("overwrite")

 # .saveAsTable: grava como tabela gerenciada no catálogo, com nome em três níveis:
 # catálogo.schema.tabela -> olist.default.abt_ativacao_pati
 # O formato padrão no Databricks é Delta.
 .saveAsTable("olist.default.abt_ativacao_pati"))

In [0]:
# isso é um dataframe

df = spark.table("olist.default.abt_ativacao_pati").toPandas()

# isso é uma string

target = 'flChurnMes'


In [0]:
df

In [0]:
df.head()

In [0]:

target = 'flChurn'

# isso é uma lista


colunas = df.columns.tolist()

features = colunas[5:]

In [0]:
df[features]

In [0]:
print(df.columns.tolist())

In [0]:
df = spark.table("olist.default.abt_ativacao_pati").toPandas()

df['flChurnMes'] = df['flagChurn']
df['flChurn'] = df['flagChurn']

target = 'flChurnMes'

In [0]:
df[target]

In [0]:
x = df[features] # isso é uma matriz (dataframe)

y = df[target] # isso é um vetor (series)

In [0]:
type(x)

In [0]:
type(y)

In [0]:
## separando treino de teste

# Importa o módulo model_selection do scikit-learn, que reúne ferramentas
# para dividir dados e validar modelos.
from sklearn import model_selection

# train_test_split divide x (features) e y (alvo) em dois blocos:
#   treino: o modelo aprende com ele
#   teste: o modelo é avaliado em dados que nunca viu
# Retorna 4 objetos, na ordem: x_train, x_test, y_train, y_test.
x_train, x_test, y_train, y_test = model_selection.train_test_split(
    x, y,

    # test_size=0.2: 20% das linhas vão para teste e 80% para treino.
    test_size=0.2,

    # random_state=42: fixa a semente do sorteio. Assim a divisão é sempre
    # a mesma a cada execução, o que torna os resultados reproduzíveis.
    # (42 é só uma convenção; qualquer número serve.)
    random_state=42,

    # stratify=y: divisão estratificada. Mantém em treino e teste a mesma
    # proporção de classes de y. Se 30% dos vendedores são churn na base,
    # serão ~30% no treino e ~30% no teste.
    stratify=y
)

In [0]:
x_train

In [0]:
x_test

In [0]:
y_train

In [0]:
y_test

In [0]:
print("taxa de resposta na base de treino: ", y_train.mean())
print("taxa de resposta na base de teste: ", y_test.mean())
## criando o modelo

